# TinyDoc-VLM 768 Final Eval (Kaggle)

Runs training/eval_768.py against the step-8000 checkpoint from
eulogik/TinyDoc-VLM-768-checkpoints with fresh synthetic pages.
Results are uploaded to the runtime repo and printed to the log.

In [ ]:
import subprocess, sys, os, time

REPO_URL = 'https://github.com/eulogik/TinyDoc-VLM'
REPO = '/kaggle/working/tinydoc-vlm'

PAGES = os.environ.get('PAGES', '12')
SEED = os.environ.get('SEED', '20260814')

if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        _c = UserSecretsClient()
        for _ in range(5):
            try:
                os.environ['HF_TOKEN'] = _c.get_secret('HF_TOKEN')
                break
            except Exception:
                time.sleep(5)
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN not set. Add it to Kaggle Secrets (Settings > Secrets).')

# Fresh clone (kaggle kernels reuse /kaggle/working between reruns)
if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO])

# Deps (mirror kaggle_train.py's proven list)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==5.12.1', 'sentencepiece', 'tokenizers', 'pillow',
                'numpy', 'pandas', 'tqdm', 'pyyaml', 'einops', 'faker', 'jinja2',
                'pydantic', 'datasets', 'accelerate', 'huggingface_hub'])

# Torch pin: image torch 2.10.0 native-SIGSEGVs on T4; pin Colab-proven 2.6.0.
try:
    import torch
    cap = torch.cuda.get_device_capability(0)
    extra = 'cu118' if cap[0] < 7 else 'cu124'
    ver = torch.__version__.split('+')[0]
    if ver != '2.6.0':
        print(f'GPU cap={cap}; pinning torch 2.6.0+{extra} ...')
        ok1 = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                              f'torch==2.6.0+{extra}', f'torchvision==0.21.0+{extra}'],
                             capture_output=True).returncode == 0
        ok2 = ok1 or subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                                     f'torch==2.6.0+{extra}', f'torchvision==0.21.0+{extra}',
                                     '--extra-index-url', 'https://download.pytorch.org/whl/' + extra],
                                    capture_output=True).returncode == 0
        if ok2:
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchaudio'])
        else:
            print('WARNING: torch pin FAILED; continuing on image torch')
except Exception as e:
    print('torch pin skipped:', e)

# Run the eval (downloads step-8000 checkpoint from the private hub repo)
out_path = '/kaggle/working/eval_768_results.json'
p = subprocess.run(
    [sys.executable, 'training/eval_768.py',
     '--device', 'cuda', '--pages', PAGES, '--seed', SEED,
     '--max-new-tokens', '256', '--output', out_path],
    cwd=REPO,
)

# Upload results to the runtime repo + print summary (visible in kernel log)
try:
    from huggingface_hub import upload_file
    upload_file(path_or_fileobj=out_path, path_in_repo='eval/eval_768_results.json',
                repo_id='eulogik/TinyDoc-VLM-runtime', repo_type='model',
                token=os.environ['HF_TOKEN'])
    print('Results uploaded to eulogik/TinyDoc-VLM-runtime/eval/eval_768_results.json')
except Exception as e:
    print('Results upload failed:', e)

import json as _json
try:
    d = _json.load(open(out_path))
    print('=== SUMMARY ===')
    print('checkpoint_step:', d.get('checkpoint_step'))
    print('pages:', d.get('pages'))
    print('qa_accuracy:', d.get('qa_accuracy'), f"({d.get('qa_correct')}/{d.get('qa_total')})")
    print('=== FULL RESULTS ===')
    print(_json.dumps(d, indent=2, default=str))
except Exception as e:
    print('Could not read results:', e)

sys.exit(p.returncode)